In [ ]:
import os
import math
import pprint
import re
from dotenv import load_dotenv
from datetime import date, timedelta, datetime
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth

load_dotenv()

pd.options.display.max_columns = None  # type: ignore
pd.options.display.float_format = "{:,.2f}".format

token = os.getenv("PERSONAL_ACCESS_TOKEN")

In [ ]:
# CONFIG
org = "Dtek-SP"
project = "MarketRMS"
pat = token
auth=HTTPBasicAuth('', pat)
base = f"https://analytics.dev.azure.com/{org}/{project}/_odata/v3.0-preview"
base_url = f"https://dev.azure.com/{org}"
snapshot_path = "work_item_snapshot.parquet"
today = date.today()

# Download live snapshot

In [ ]:
from ado_live_snapshot import download_live_snapshot

wi_live = download_live_snapshot(org, project, auth)

# Download Shapshots history

In [ ]:
# Request for new work items

# identify dates to query

existing_snapshots = pd.read_parquet(snapshot_path)
existing_snapshots["SnapshotDate"] = pd.to_datetime(existing_snapshots["SnapshotDate"], utc=True, errors="coerce")
last_snapshot_date = existing_snapshots["SnapshotDate"].max().date()


start = last_snapshot_date + timedelta(days=1)
# ADO Analytics contains data for today. But it may contain yesterday's data till today is actually ended. So we query only for yesterday.
end   = today - timedelta(days=1)

if start > end:
    print("No new days to load – data is up to date.")
    raise SystemExit
else:
    print(f"Will load snapshots from {start} to {end}")

# make a query

wi_url = f"{base}/WorkItemSnapshot"

all_new_rows = []

cur = start
while cur <= end:
    next_day = cur + timedelta(days=1)

    wi_params = {
        "$select": "DateValue,ProjectSK,WorkItemId,WorkItemType,Title,State,StateCategory,StateChangeDate,Priority,TagNames,ParentWorkItemId,RevisedDate,AssignedToUserSK,ChangedByUserSK,CreatedByUserSK,OriginalEstimate,CompletedWork,RemainingWork,StoryPoints,WorkItemRevisionSK,AreaSK,IterationSK,ChangedDate,CreatedDate,ClosedDate,InProgressDate,CompletedDate",
        "$filter": f"DateValue ge {cur.isoformat()} and DateValue lt {next_day.isoformat()}",
        "$count": "true"
    }

    url = wi_url
    params = wi_params

    while url:
        resp = requests.get(url, auth=auth, params=params)
        resp.raise_for_status()
        data = resp.json()
        all_new_rows.extend(data["value"])

        url = data.get("@odata.nextLink")
        params = None  # nextLink already has query if present

    print(f"Loaded snapshot for {cur} (rows so far: {len(all_new_rows)})")
    cur = next_day

new_snapshots = pd.DataFrame(all_new_rows)
print("New rows loaded:", len(new_snapshots))

In [ ]:
wi_dates = [
    "DateValue", "ChangedDate", "CreatedDate",
    "ClosedDate", "InProgressDate", "CompletedDate", "StateChangeDate"
]

for col in wi_dates:
    new_snapshots[col] = pd.to_datetime(
        new_snapshots[col],
        utc=True,
        errors="coerce",     # bad/empty values -> NaT instead of crash
        format="ISO8601"     # handles with/without milliseconds
    )

new_snapshots = new_snapshots.rename(columns = {"DateValue": "SnapshotDate"})

In [ ]:
# Request for users
users_url = f"{base}/Users"
users_params = {
    # "$top": 10,
    "$select": "UserSK,UserName,UserEmail"
    }
resp_users = requests.get(users_url, auth=auth, params=users_params)
resp_users.raise_for_status()
users_df = pd.DataFrame(resp_users.json()["value"])

# Request for areas
areas_url = f"{base}/Areas"
areas_params = {
    # "$top": 10,
    "$select": "AreaSK,AreaName,AreaPath"
    }
resp_areas = requests.get(areas_url, auth=auth, params=areas_params)
resp_areas.raise_for_status()
areas_df = pd.DataFrame(resp_areas.json()["value"])

# Request for iterations
iters_url = f"{base}/Iterations"
iters_params = {
    "$select": "IterationSK,IterationName,IterationPath,StartDate,EndDate"
    }
resp_iter = requests.get(iters_url, auth=auth, params=iters_params)
resp_iter.raise_for_status()
iters_df = pd.DataFrame(resp_iter.json()["value"])
iters_dates = [
    "StartDate", "EndDate"
]

for date in iters_dates:
    iters_df[date] = pd.to_datetime(iters_df[date], utc=True)

# Request for projects
projects_url = f"{base}/Projects"
projects_params = {
    "$select": "ProjectSK,ProjectName"
    }
resp_projects = requests.get(projects_url, auth=auth, params=projects_params)
resp_projects.raise_for_status()
projects_df = pd.DataFrame(resp_projects.json()["value"])


In [ ]:
# take metadata from SK columns

user_sk_map = {
    "AssignedToUserSK": "AssignedTo",
    "ChangedByUserSK": "ChangedBy",
    "CreatedByUserSK": "CreatedBy"
}

new_snapshots = new_snapshots.copy()

for sk_col, name_col in user_sk_map.items():
    new_snapshots = (
        new_snapshots
        .merge(users_df[["UserSK", "UserName"]],
               left_on=sk_col, right_on="UserSK", how="left")
        .rename(columns={"UserName": name_col})
        .drop(columns=[sk_col, "UserSK"])
    )

new_snapshots = (
    new_snapshots
    .merge(areas_df, on="AreaSK", how="left")
    .drop(columns=["AreaSK"])
    .merge(iters_df, on="IterationSK", how="left")
    .rename(columns={"StartDate": "IterationStart", "EndDate": "IterationEnd"})
    .drop(columns=["IterationSK"])
    .merge(projects_df, on="ProjectSK", how="left")
    .drop(columns=["ProjectSK"])
)

new_snapshots["WorkItemURL"] = (
    base_url
    + "/"
    + new_snapshots["ProjectName"].astype(str)
    + "/_workitems/edit/"
    + new_snapshots["WorkItemId"].astype(str)
)

In [ ]:
# Write new data
work_items =  pd.concat([existing_snapshots, new_snapshots], ignore_index=True)

# Just in case
work_items = work_items.drop_duplicates(
    subset=["WorkItemId", "SnapshotDate"],
    keep="last"
)

work_items.to_parquet(snapshot_path, index=False)
print("Parquet updated. Total rows now:", len(work_items))